In [1]:

from google.colab import drive
import torch
import os
import sys

# Connect Google Drive
drive.mount('/content/drive')

# Project directory
PROJECT_ROOT = "/content/drive/MyDrive/YOLOv8_Fault_Tolerance"
CODE_DIR = os.path.join(PROJECT_ROOT, "code")

print("=" * 50)
print("YOLOv8s Fault Tolerance - Notebook 03")
print("=" * 50)

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Project exists:", os.path.isdir(PROJECT_ROOT))
print("Code directory exists:", os.path.isdir(CODE_DIR))


Mounted at /content/drive
YOLOv8s Fault Tolerance - Notebook 03
Python version: 3.13.16
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Project exists: True
Code directory exists: True


In [2]:

import os
import importlib.metadata

print("=" * 50)
print("Notebook 03 - Baseline Preflight Check")
print("=" * 50)

# Check Ultralytics version
try:
    version = importlib.metadata.version("ultralytics")
    print("Ultralytics version:", version)
except importlib.metadata.PackageNotFoundError:
    print("Ultralytics: NOT INSTALLED")

# Check existing model weight files
for folder in [
    CODE_DIR,
    os.path.join(CODE_DIR, "weights"),
    os.path.join(PROJECT_ROOT, "artifacts"),
]:
    print(f"\nChecking: {folder}")

    if os.path.isdir(folder):
        files = sorted(os.listdir(folder))
        print("Contents:", files[:25])
    else:
        print("Directory does not exist")


Notebook 03 - Baseline Preflight Check
Ultralytics: NOT INSTALLED

Checking: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code
Contents: ['.git', '.gitignore', 'README.md', 'configs', 'data', 'evaluation', 'experiments', 'fault_injection', 'fault_tolerance', 'losses', 'models', 'notebooks', 'requirements.txt', 'results', 'tests', 'training']

Checking: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/weights
Directory does not exist

Checking: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/artifacts
Contents: []


In [3]:

%pip install ultralytics==8.4.189


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [4]:

import ultralytics

print("Ultralytics version:", ultralytics.__version__)
ultralytics.checks()


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.6/112.6 GB disk)


In [5]:

from ultralytics import YOLO
import torch

# Load the pretrained YOLOv8s model
model = YOLO("yolov8s.pt")

print("=" * 50)
print("YOLOv8s Baseline - Model Verification")
print("=" * 50)

print("Model task:", model.task)
print("Model type:", type(model.model).__name__)
print("Number of classes:", len(model.names))

parameters = sum(p.numel() for p in model.model.parameters())
print(f"Total parameters: {parameters:,}")

print("CUDA available:", torch.cuda.is_available())
print("Model loaded successfully!")


YOLOv8s Baseline - Model Verification
Model task: detect
Model type: DetectionModel
Number of classes: 80
Total parameters: 11,166,560
CUDA available: True
Model loaded successfully!


In [6]:

from pathlib import Path

dataset_root = Path(PROJECT_ROOT) / "datasets"

print("=" * 50)
print("Dataset Inventory")
print("=" * 50)

print("Dataset directory exists:", dataset_root.exists())

if dataset_root.exists():
    for item in sorted(dataset_root.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}")
            for child in sorted(item.iterdir())[:15]:
                print(f"       {child.name}")
        else:
            size_mb = item.stat().st_size / (1024 ** 2)
            print(f"[FILE] {item.name} ({size_mb:.2f} MB)")


Dataset Inventory
Dataset directory exists: True


In [7]:

from ultralytics import YOLO
from pathlib import Path

# Quick dataset pipeline check (not final evaluation)
model = YOLO("yolov8s.pt")

results = model.val(
    data="coco8.yaml",
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    project=str(Path(CODE_DIR) / "results" / "baseline_validation"),
    name="coco8_smoke_test",
    exist_ok=True
)

print("=" * 50)
print("COCO8 Smoke Test Results")
print("=" * 50)
print(f"mAP50: {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 72 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 3.1Kfiles/s 0.0s
Dataset download success ✅ (0.2s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1083.5±517.5 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 110.4it/s 0.0s
val: New cache created: /content/datasets/coco8/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.0it/s 0.3s
                   all          4         17       0.87       0.92      0.951      0.719
                person          3         10      0.855       0.59    

In [8]:

import shutil
from pathlib import Path

project_datasets = Path(PROJECT_ROOT) / "datasets"

drive_usage = shutil.disk_usage(project_datasets)
local_usage = shutil.disk_usage("/content")

print("Google Drive free space:",
      round(drive_usage.free / (1024**3), 2), "GB")

print("Colab local free space:",
      round(local_usage.free / (1024**3), 2), "GB")

print("Dataset destination:", project_datasets)


Google Drive free space: 10.73 GB
Colab local free space: 70.01 GB
Dataset destination: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets


In [9]:

from pathlib import Path
import subprocess

COCO_ROOT = Path("/content/datasets/coco2017")
COCO_ROOT.mkdir(parents=True, exist_ok=True)

zip_path = COCO_ROOT / "val2017.zip"
images_dir = COCO_ROOT / "images"
images_dir.mkdir(exist_ok=True)

url = "http://images.cocodataset.org/zips/val2017.zip"

print("Downloading COCO 2017 validation images...")

if not zip_path.exists():
    subprocess.run(
        ["wget", "-c", "-O", str(zip_path), url],
        check=True
    )
else:
    print("ZIP already exists; skipping download.")

print("Extracting images...")

subprocess.run(
    ["unzip", "-n", "-q", str(zip_path), "-d", str(images_dir)],
    check=True
)

image_files = list((images_dir / "val2017").glob("*.jpg"))

print("=" * 50)
print("COCO 2017 Validation")
print("=" * 50)
print("Images found:", len(image_files))
print("Dataset path:", images_dir / "val2017")


Extracting images...
COCO 2017 Validation
Images found: 5000
Dataset path: /content/datasets/coco2017/images/val2017


In [10]:

from pathlib import Path
import subprocess
import json

COCO_ROOT = Path("/content/datasets/coco2017")
annotations_dir = COCO_ROOT / "annotations"
annotations_dir.mkdir(parents=True, exist_ok=True)

zip_path = COCO_ROOT / "annotations_trainval2017.zip"
url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

print("Downloading COCO 2017 annotations...")

if not zip_path.exists():
    subprocess.run(
        ["wget", "-c", "-O", str(zip_path), url],
        check=True
    )

subprocess.run(
    ["unzip", "-n", "-q", str(zip_path),
     "annotations/instances_val2017.json",
     "-d", str(COCO_ROOT)],
    check=True
)

json_path = annotations_dir / "instances_val2017.json"

with open(json_path, "r") as f:
    coco = json.load(f)

print("=" * 50)
print("COCO 2017 Annotation Verification")
print("=" * 50)
print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", len(coco["categories"]))
print("Annotation file:", json_path)


COCO 2017 Annotation Verification
Images: 5000
Annotations: 36781
Categories: 80
Annotation file: /content/datasets/coco2017/annotations/instances_val2017.json


In [11]:

import json
from pathlib import Path
from collections import Counter

COCO_ROOT = Path("/content/datasets/coco2017")
json_path = COCO_ROOT / "annotations" / "instances_val2017.json"
labels_dir = COCO_ROOT / "labels" / "val2017"
labels_dir.mkdir(parents=True, exist_ok=True)

with open(json_path, "r") as f:
    coco = json.load(f)

# Official COCO category IDs in YOLO class order
coco91_to_coco80 = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    22, 23, 24, 25, 27, 28, 31, 32, 33, 34,
    35, 36, 37, 38, 39, 40, 41, 42, 43, 44,
    46, 47, 48, 49, 50, 51, 52, 53, 54, 55,
    56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
    67, 70, 72, 73, 74, 75, 76, 77, 78, 79,
    80, 81, 82, 84, 85, 86, 87, 88, 89, 90
]

category_map = {
    category_id: index
    for index, category_id in enumerate(coco91_to_coco80)
}

assert len(category_map) == 80
assert set(category_map) == {
    c["id"] for c in coco["categories"]
}

images = {img["id"]: img for img in coco["images"]}
labels = {img_id: [] for img_id in images}
stats = Counter()

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        stats["crowd_skipped"] += 1
        continue

    img = images[ann["image_id"]]
    x, y, w, h = ann["bbox"]

    if w <= 0 or h <= 0:
        stats["invalid_boxes"] += 1
        continue

    W, H = img["width"], img["height"]
    xc = (x + w / 2) / W
    yc = (y + h / 2) / H
    wn = w / W
    hn = h / H

    class_id = category_map[ann["category_id"]]

    labels[ann["image_id"]].append(
        f"{class_id} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}"
    )
    stats["boxes_written"] += 1

for img_id, img in images.items():
    label_path = labels_dir / (
        Path(img["file_name"]).stem + ".txt"
    )
    label_path.write_text(
        "\n".join(labels[img_id]) +
        ("\n" if labels[img_id] else "")
    )

print("=" * 50)
print("COCO to YOLO Conversion")
print("=" * 50)
print("Label files:", len(list(labels_dir.glob("*.txt"))))
print("Boxes written:", stats["boxes_written"])
print("Crowd annotations skipped:", stats["crowd_skipped"])
print("Invalid boxes:", stats["invalid_boxes"])
print("Output:", labels_dir)


COCO to YOLO Conversion
Label files: 5000
Boxes written: 36335
Crowd annotations skipped: 446
Invalid boxes: 0
Output: /content/datasets/coco2017/labels/val2017


In [12]:

from pathlib import Path
import numpy as np

labels_dir = Path(
    "/content/datasets/coco2017/labels/val2017"
)

total_boxes = 0
invalid_lines = []
empty_files = 0

for label_file in sorted(labels_dir.glob("*.txt")):
    lines = label_file.read_text().splitlines()

    if not lines:
        empty_files += 1

    for line_number, line in enumerate(lines, start=1):
        try:
            values = list(map(float, line.split()))

            if len(values) != 5:
                raise ValueError("Expected 5 values")

            cls, x, y, w, h = values

            valid = (
                cls.is_integer()
                and 0 <= cls < 80
                and np.isfinite(values).all()
                and 0 <= x <= 1
                and 0 <= y <= 1
                and 0 < w <= 1
                and 0 < h <= 1
                and x - w / 2 >= -1e-6
                and y - h / 2 >= -1e-6
                and x + w / 2 <= 1 + 1e-6
                and y + h / 2 <= 1 + 1e-6
            )

            if not valid:
                invalid_lines.append(
                    (label_file.name, line_number, line)
                )

            total_boxes += 1

        except (ValueError, TypeError):
            invalid_lines.append(
                (label_file.name, line_number, line)
            )

print("=" * 50)
print("YOLO Label Quality Check")
print("=" * 50)
print("Total label files:",
      len(list(labels_dir.glob("*.txt"))))
print("Total boxes:", total_boxes)
print("Empty label files:", empty_files)
print("Invalid lines:", len(invalid_lines))
print("First 5 invalid examples:", invalid_lines[:5])


YOLO Label Quality Check
Total label files: 5000
Total boxes: 36335
Empty label files: 48
Invalid lines: 0
First 5 invalid examples: []


In [13]:

from pathlib import Path
import yaml

COCO_ROOT = Path("/content/datasets/coco2017")

# Use the class names from the loaded YOLOv8s model
class_names = [model.names[i] for i in range(80)]

dataset_config = {
    "path": str(COCO_ROOT),
    "train": "images/val2017",
    "val": "images/val2017",
    "names": dict(enumerate(class_names)),
}

yaml_path = COCO_ROOT / "coco2017_val.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(
        dataset_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print("=" * 50)
print("Dataset Configuration")
print("=" * 50)
print("YAML path:", yaml_path)
print("Validation images:",
      COCO_ROOT / dataset_config["val"])
print("Number of classes:",
      len(dataset_config["names"]))
print("First 5 classes:", class_names[:5])
print("YAML created:", yaml_path.exists())


Dataset Configuration
YAML path: /content/datasets/coco2017/coco2017_val.yaml
Validation images: /content/datasets/coco2017/images/val2017
Number of classes: 80
First 5 classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane']
YAML created: True


In [14]:

from pathlib import Path
import time

print("=" * 50)
print("YOLOv8s - COCO 2017 Baseline Validation")
print("=" * 50)

start_time = time.time()

metrics = model.val(
    data="/content/datasets/coco2017/coco2017_val.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    max_det=300,
    project=str(Path(CODE_DIR) / "results" / "baseline_validation"),
    name="coco2017_val",
    exist_ok=True,
    plots=True
)

elapsed = time.time() - start_time

print("=" * 50)
print("BASELINE VALIDATION RESULTS")
print("=" * 50)
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Total validation time: {elapsed:.1f} seconds")
print("Results directory:", metrics.save_dir)


YOLOv8s - COCO 2017 Baseline Validation
Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 72 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1466.0±934.2 MB/s, size: 158.3 KB)
val: Scanning /content/datasets/coco2017/labels/val2017... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 2.0Kit/s 2.6s
val: New cache created: /content/datasets/coco2017/labels/val2017.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 4.9it/s 1:04
                   all       5000      36335      0.684      0.562      0.611      0.444
                person       2693      10777      0.792      0.715      0.793      0.568
               bicycle        149        314      0.712      0.449      0.566       0.34
                   car        535       1918      0.719      0.595      0.656      0.438
         

In [15]:

import json
from pathlib import Path
from datetime import datetime, timezone
import ultralytics
import torch

results_dir = (
    Path(CODE_DIR) / "results" / "baseline_validation"
)
results_dir.mkdir(parents=True, exist_ok=True)

baseline_record = {
    "experiment": "yolov8s_coco2017_baseline",
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model": "yolov8s.pt",
    "dataset": "COCO 2017 val",
    "num_images": 5000,
    "num_ground_truth_boxes_yolo": 36335,
    "evaluation_method": "Ultralytics YOLO labels",
    "official_cocoeval": False,
    "settings": {
        "imgsz": 640,
        "batch": 16,
        "device": 0,
        "conf": 0.001,
        "iou": 0.7,
        "max_det": 300
    },
    "environment": {
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0)
    },
    "metrics": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map)
    },
    "validation_output": str(metrics.save_dir)
}

record_path = (
    results_dir / "baseline_coco2017_metrics.json"
)

with open(record_path, "w") as f:
    json.dump(
        baseline_record,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Baseline record saved:", record_path)
print("File exists:", record_path.exists())
print("Recorded mAP50-95:",
      baseline_record["metrics"]["map50_95"])
print("Recorded Ultralytics version:",
      baseline_record["environment"]["ultralytics"])


Baseline record saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/baseline_validation/baseline_coco2017_metrics.json
File exists: True
Recorded mAP50-95: 0.4443139167528017
Recorded Ultralytics version: 8.4.189


In [16]:

from pathlib import Path
import hashlib
import json

weights_path = Path("yolov8s.pt").resolve()

assert weights_path.is_file(), (
    f"Model weights not found: {weights_path}"
)

sha256 = hashlib.sha256()

with open(weights_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

weights_hash = sha256.hexdigest()

record_path = (
    Path(CODE_DIR)
    / "results"
    / "baseline_validation"
    / "baseline_coco2017_metrics.json"
)

with open(record_path, "r") as f:
    record = json.load(f)

record["model_weights"] = {
    "filename": weights_path.name,
    "sha256": weights_hash,
    "size_bytes": weights_path.stat().st_size
}

with open(record_path, "w") as f:
    json.dump(record, f, indent=2, ensure_ascii=False)

print("=" * 50)
print("Model Weights Verification")
print("=" * 50)
print("Weights path:", weights_path)
print("SHA-256:", weights_hash)
print("Size (MB):", round(weights_path.stat().st_size / 1024**2, 2))
print("Baseline record updated:", record_path.exists())


Model Weights Verification
Weights path: /content/yolov8s.pt
SHA-256: 1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a
Size (MB): 21.54
Baseline record updated: True


In [17]:

import subprocess
from pathlib import Path

repo_dir = Path(CODE_DIR)

result = subprocess.run(
    ["git", "status", "--short", "--untracked-files=normal"],
    cwd=repo_dir,
    capture_output=True,
    text=True
)

print("=" * 50)
print("Git Working Tree Status")
print("=" * 50)

if result.returncode == 0:
    print(result.stdout or "Working tree is clean.")
else:
    print("Git error:", result.stderr)

print("=" * 50)
print("Notebook 03 exists:",
      (repo_dir / "notebooks" /
       "03_baseline_validation.ipynb").exists())


Git Working Tree Status
 M notebooks/02_baseline_benchmark.ipynb

Notebook 03 exists: False


In [18]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

matches = list(
    drive_root.rglob("03_baseline_validation.ipynb")
)

print("=" * 50)
print("Notebook 03 Location Check")
print("=" * 50)

print("Number of matches:", len(matches))

for path in matches:
    print("Found:", path)

expected_path = (
    Path(CODE_DIR)
    / "notebooks"
    / "03_baseline_validation.ipynb"
)

print("Expected Git path:", expected_path)
print("Exists at expected path:", expected_path.exists())


Notebook 03 Location Check
Number of matches: 1
Found: /content/drive/MyDrive/Colab Notebooks/03_baseline_validation.ipynb
Expected Git path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/notebooks/03_baseline_validation.ipynb
Exists at expected path: False
